# FX spot and forwards

**Prerequisites:** **06**. **`fx_spot`** and **`fx_forward`** with **`FxMatrix`** quotes patched into the JSON snapshot (Python `to_json()` may emit empty `fx.quotes`).


## Concept

- **Spot:** immediate exchange of notionals at the dealt rate.
- **Forward:** covered interest parity links FX forward to domestic/foreign discount curves.
- **Multi-currency exposure:** notionals are tagged with **`currency`**; PV lands in the instrument’s settlement currency.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.currency import Currency
from finstack.core.market_data import DiscountCurve, FxMatrix, MarketContext

print("Imports OK (FX).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

fx_spot = json.dumps(
    {
        "type": "fx_spot",
        "spec": {
            "id": "EURUSD-SPOT",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "notional": {"amount": "1000000", "currency": "EUR"},
            "spot_rate": 1.10,
            "settlement_lag_days": 2,
            "bdc": "modified_following",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

fx_fwd = json.dumps(
    {
        "type": "fx_forward",
        "spec": {
            "id": "EURUSD-FWD-6M",
            "base_currency": "EUR",
            "quote_currency": "USD",
            "notional": {"amount": "1000000", "currency": "EUR"},
            "maturity": "2025-07-15",
            "contract_rate": 1.12,
            "domestic_discount_curve_id": "USD-OIS",
            "foreign_discount_curve_id": "EUR-OIS",
            "attributes": {"tags": ["fx"], "meta": {"pair": "EURUSD"}},
            "pricing_overrides": {},
        },
    }
)

for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    try:
        validate_instrument_json(js)
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois_usd = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
ois_eur = DiscountCurve(
    "EUR-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.988), (1.0, 0.975), (3.0, 0.92), (5.0, 0.85), (10.0, 0.70)],
    day_count="act_365f",
)
fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
mc = MarketContext().insert(ois_usd).insert(ois_eur)
mc.insert_fx(fx)
state = json.loads(mc.to_json())
state["fx"]["quotes"] = [["EUR", "USD", 1.10]]
market_json = json.dumps(state)
print("fx quotes patched:", state["fx"]["quotes"])


## Pricing


In [ ]:
for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    try:
        out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


## Metrics


In [ ]:
for label, js in (("fx_spot", fx_spot), ("fx_forward", fx_fwd)):
    try:
        out = price_instrument_with_metrics(
            js, market_json, AS_OF_STR, model="discounting", metrics=["fx01", "fx_delta", "forward_points"]
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in ("fx01", "fx_delta", "forward_points"):
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)


## Takeaways

- **Patch `fx.quotes`** when snapshots serialize empty but FX instruments need a spot triangle.
- **Two discount curves** anchor forward FX via interest parity.
- FX metrics (`fx01`, `fx_delta`, …) are registry-dependent.
